In [ ]:
"""Day 27 / Session 1 - Lab Assignment 2: Top-k and Top-p (Nucleus) Filtering - Truncating the Sampling Distribution."""
import numpy as np

logits = np.array([3.0, 2.0, 1.0, 0.0, -1.0])         # 5-token vocab, already in descending order


def softmax(logits):
    z = np.asarray(logits, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()


def top_k_mask(probs, k):
    probs = np.asarray(probs, dtype=float)
    k = int(min(max(k, 0), len(probs)))                       # clamp k to the vocab size
    mask = np.zeros(len(probs), dtype=bool)
    mask[np.argsort(-probs, kind="stable")[:k]] = True        # the k highest-probability tokens
    return mask


def top_k_filter(probs, k):
    probs = np.asarray(probs, dtype=float)
    mask = top_k_mask(probs, max(int(k), 1))                  # always keep at least one token
    out = np.where(mask, probs, 0.0)
    return out / out.sum()                                    # renormalise; ratios between kept tokens are unchanged


def _nucleus_count(probs, p_thresh):
    probs = np.asarray(probs, dtype=float)
    order = np.argsort(-probs, kind="stable")                 # highest probability first
    cumulative = np.cumsum(probs[order])
    n = int(np.searchsorted(cumulative, p_thresh - 1e-12) + 1)    # smallest prefix whose mass reaches p_thresh
    return int(min(max(n, 1), len(probs))), order             # always at least one token


def top_p_filter(probs, p_thresh):
    probs = np.asarray(probs, dtype=float)
    n, order = _nucleus_count(probs, p_thresh)
    out = np.zeros_like(probs)
    out[order[:n]] = probs[order[:n]]                         # keep the nucleus, zero the tail
    return out / out.sum()


def nucleus_size(probs, p_thresh):
    return _nucleus_count(probs, p_thresh)[0]                 # how many tokens top-p keeps (int >= 1)
